# Imbalanced Learning: Class Weights, Resampling by Weight and Thresholds

**Dataset:** `telco_churn` | **Level:** advanced

Does reweighting help more than moving the decision threshold?

Sections: 1 Problem, 2 Method, 3 Setup and data, 4 Experiment, 5 Results, 6 Interpretation, 7 Limitations, 8 Exercises, 9 References.

## 1. Problem

Churn is the minority class (about 26.5 percent). Accuracy at threshold 0.5 hides poor recall. We test three ways to deal with imbalance and compare them on ranking and on F1.

## 2. Method

(a) plain model, (b) class weights (`class_weight=balanced`), (c) plain model with a threshold chosen on validation rows to maximise F1. The ranking metric PR-AUC is not changed by a threshold, so any PR-AUC difference comes from the model itself. No synthetic oversampling is used because it must be done inside each cross validation fold and adds a dependency.

## 3. Setup and data

Telco customer churn (7043 rows, 21 columns, about 26.5 percent churn). Source and checksum are documented in `datasets/catalog/supervised.json` and `01_supervised_learning/` notebooks.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from ml_lab.data import load_dataset
from ml_lab.preprocessing import split_three_way, make_model_pipeline
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz
NB_ID = "imbalanced_learning"
SEED = 42
set_seed(SEED); viz.apply_style(); print_environment()

python 3.13.18 | Linux-6.18.44-fc-v595-x86_64-with-glibc2.39
  numpy==2.5.3
  pandas==3.0.5
  scipy==1.18.1
  scikit-learn==1.9.1
  matplotlib==3.11.2
  xgboost==3.4.1
  lightgbm==4.7.0
  catboost==1.2.10
  umap-learn==0.6.0
  torch==2.15.0
  gymnasium==1.4.0


In [2]:
b = load_dataset("telco_churn"); X, y = b.X, b.y
Xtr, Xva, Xte, ytr, yva, yte = split_three_way(X, y, task="classification", seed=SEED)
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, brier_score_loss
print(len(Xtr), len(Xva), len(Xte), "| churn share", round(float(y.mean()), 3))

4225 1409 1409 | churn share 0.265


## 4. Experiment

Fit three logistic regression variants on training rows and a random forest with and without class weights.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
variants = {"logistic plain": LogisticRegression(max_iter=2000), "logistic balanced": LogisticRegression(max_iter=2000, class_weight="balanced"),
            "forest plain": RandomForestClassifier(300, min_samples_leaf=5, n_jobs=2, random_state=SEED),
            "forest balanced": RandomForestClassifier(300, min_samples_leaf=5, class_weight="balanced", n_jobs=2, random_state=SEED)}
fit = {n: make_model_pipeline(Xtr, m, scale="logistic" in n).fit(Xtr, ytr) for n, m in variants.items()}
def best_threshold(p, y_true):
    grid = np.linspace(0.05, 0.9, 86); f = [f1_score(y_true, p >= t) for t in grid]; return float(grid[int(np.argmax(f))])
print("fitted", list(fit))

fitted ['logistic plain', 'logistic balanced', 'forest plain', 'forest balanced']


## 5. Results

Test rows, scored once. F1 is reported at threshold 0.5 and at the threshold tuned on validation rows.

In [4]:
rows = []
for n, m in fit.items():
    pv, pt = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    t = best_threshold(pv, yva)
    rows.append({"model": n, "PR_AUC": average_precision_score(yte, pt), "F1_at_0.5": f1_score(yte, pt >= 0.5), "tuned_threshold": t, "F1_at_tuned": f1_score(yte, pt >= t), "recall_at_0.5": ((pt >= 0.5) & (yte == 1)).sum() / (yte == 1).sum()})
tab = pd.DataFrame(rows).set_index("model"); display(tab.round(3))
METRICS = {f"{n.replace(' ', '_')}_{k}": float(v) for n, r in tab.iterrows() for k, v in r.items()}
CONFIG = {"threshold_grid": "0.05 to 0.9 on validation"}; EXTRA = {"table": tab.round(4).reset_index().to_dict("records")}; ALGORITHM = "class weights and threshold tuning"

record = ExperimentRecord(notebook=NB_ID, family="advanced", algorithm=ALGORITHM, dataset="telco_churn", task="classification", seed=SEED, config=CONFIG, metrics=METRICS, extra=EXTRA)
print("saved:", record.save())

,PR_AUC,F1_at_0.5,tuned_threshold,F1_at_tuned,recall_at_0.5
model,,,,,
logistic plain,0.634,0.598,0.38,0.617,0.553
logistic balanced,0.633,0.615,0.61,0.616,0.783
forest plain,0.643,0.546,0.36,0.625,0.471
forest balanced,0.641,0.623,0.61,0.606,0.767


saved: experiments/results/imbalanced_learning.json


## 6. Interpretation

Read the table by column. If PR-AUC is nearly equal across plain and balanced variants while F1 at 0.5 differs, weighting acts mostly like a threshold shift. The tuned threshold column shows how much of the gain a simple threshold gives without changing the model.

## 7. Limitations

One dataset, one split, one seed. The differences are single run values with no interval. No resampling methods (SMOTE and similar) were tested.

## 8. Exercises

1. Repeat with five random splits and report the spread of F1.
2. Add a cost matrix (missed churn costs five times a wasted offer) and choose the threshold by expected cost.
3. Try SMOTE inside a cross validation pipeline and compare.

## 9. References

He, H. and Garcia, E. (2009), *Learning from imbalanced data*, IEEE TKDE, is the usual survey. It is **not verified in the build environment** and is listed for reading only. For model references see `research/source-register.md` (`breiman2001`).